In [ ]:
%pip install -q "transformers>=4.41.0" datasets evaluate accelerate scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00


In [ ]:
import json
import os
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


SEED = 42
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## 1. Mount Google Drive va cau hinh duong dan


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")


DATA_RELATIVE_PATH = Path("data/processed/student_voice_enriched_reviewed.csv")

PROJECT_DIR_CANDIDATES = [
    DRIVE_ROOT / "Student Voice Intelligence",
    DRIVE_ROOT / "AI thuc chien" / "Student Voice Intelligence",
    DRIVE_ROOT / "Colab Notebooks" / "Student Voice Intelligence",
]

PROJECT_DIR = None
for candidate in PROJECT_DIR_CANDIDATES:
    if (candidate / DATA_RELATIVE_PATH).exists():
        PROJECT_DIR = candidate
        break

DATA_PATH = PROJECT_DIR / DATA_RELATIVE_PATH
MODEL_ROOT = PROJECT_DIR / "outputs/models/transformer"
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

RUN_NAME = f"xlmr_sentiment_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = MODEL_ROOT / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MODEL_DIR = RUN_DIR / "model"
REPORT_DIR = RUN_DIR / "reports"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_PATH:", DATA_PATH)
print("RUN_DIR:", RUN_DIR)

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/Student Voice Intelligence
DATA_PATH: /content/drive/MyDrive/Student Voice Intelligence/data/processed/student_voice_enriched_reviewed.csv
RUN_DIR: /content/drive/MyDrive/Student Voice Intelligence/outputs/models/transformer/xlmr_sentiment_20260619_193243


## 2. Cau hinh train


In [ ]:
MODEL_NAME = "FacebookAI/xlm-roberta-base"
TEXT_COL = "text"
TARGET_COL = "sentiment_std_3class"
SPLIT_COL = "split_original"

LABELS = ["negative", "neutral", "positive"]
label2id = {label: idx for idx, label in enumerate(LABELS)}
id2label = {idx: label for label, idx in label2id.items()}

MAX_LENGTH = 192
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

print("label2id:", label2id)

label2id: {'negative': 0, 'neutral': 1, 'positive': 2}


## 3. Load data va lam sach nhe


In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)

required_cols = [TEXT_COL, TARGET_COL, SPLIT_COL]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Thieu cot bat buoc: {missing_cols}")

work_df = df[required_cols].copy()
work_df[TEXT_COL] = work_df[TEXT_COL].fillna("").astype(str).str.strip()
work_df[TARGET_COL] = work_df[TARGET_COL].fillna("").astype(str).str.strip()
work_df[SPLIT_COL] = work_df[SPLIT_COL].fillna("").astype(str).str.strip()

work_df = work_df[
    (work_df[TEXT_COL] != "")
    & (work_df[TARGET_COL].isin(LABELS))
    & (work_df[SPLIT_COL].isin(["train", "validation", "test"]))
].copy()

train_mask = work_df[SPLIT_COL] == "train"
train_df = work_df.loc[train_mask].copy()
conflict_counts = train_df.groupby(TEXT_COL)[TARGET_COL].nunique()
conflict_texts = set(conflict_counts[conflict_counts > 1].index)

if conflict_texts:
    before_rows = len(work_df)
    work_df = work_df[~((work_df[SPLIT_COL] == "train") & (work_df[TEXT_COL].isin(conflict_texts)))].copy()
    print(f"Da bo {before_rows - len(work_df)} dong train bi conflict label.")
else:
    print("Khong co conflict label trong train.")

work_df["labels"] = work_df[TARGET_COL].map(label2id).astype(int)

print("Shape:", work_df.shape)
print("Split distribution:")
display(pd.crosstab(work_df[SPLIT_COL], work_df[TARGET_COL], margins=True))

Da bo 2 dong train bi conflict label.
Shape: (49139, 4)
Split distribution:


sentiment_std_3class,negative,neutral,positive,All
split_original,,,,
test,2630,4725,2424,9779
train,9539,16394,8539,34472
validation,1314,2352,1222,4888
All,13483,23471,12185,49139


## 4. Tao Hugging Face Dataset va tokenize


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def to_hf_dataset(split_name: str) -> Dataset:
    split_df = work_df.loc[work_df[SPLIT_COL] == split_name, [TEXT_COL, "labels"]].reset_index(drop=True)
    if split_df.empty:
        raise ValueError(f"Split rong: {split_name}")
    return Dataset.from_pandas(split_df, preserve_index=False)


raw_datasets = DatasetDict(
    {
        "train": to_hf_dataset("train"),
        "validation": to_hf_dataset("validation"),
        "test": to_hf_dataset("test"),
    }
)


def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized_datasets = raw_datasets.map(tokenize_batch, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns([TEXT_COL])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized_datasets)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/34472 [00:00<?, ? examples/s]

Map:   0%|          | 0/4888 [00:00<?, ? examples/s]

Map:   0%|          | 0/9779 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 34472
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 4888
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 9779
    })
})


## 5. Khoi tao model va metric


In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 6. Train


In [ ]:
import inspect

training_args_kwargs = {
    "output_dir": str(CHECKPOINT_DIR),
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "num_train_epochs": NUM_EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": 0.06,
    "logging_steps": 50,
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "macro_f1",
    "greater_is_better": True,
    "fp16": torch.cuda.is_available(),
    "report_to": "none",
    "seed": SEED,
}

training_args_signature = inspect.signature(TrainingArguments.__init__)
if "eval_strategy" in training_args_signature.parameters:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_args_kwargs)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_datasets["train"],
    "eval_dataset": tokenized_datasets["validation"],
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}

trainer_signature = inspect.signature(Trainer.__init__)

if "processing_class" in trainer_signature.parameters:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_signature.parameters:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

train_result = trainer.train()
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

print("Da luu model vao:", MODEL_DIR)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.442684,0.390229,0.856383,0.850512,0.855329
2,0.316367,0.420409,0.859247,0.857123,0.859451
3,0.283576,0.413033,0.866612,0.863991,0.866550


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Da luu model vao: /content/drive/MyDrive/Student Voice Intelligence/outputs/models/transformer/xlmr_sentiment_20260619_193243/model


## 7. Evaluate tren validation/test va luu report


In [10]:
def evaluate_split(split_name: str) -> dict:
    predictions = trainer.predict(tokenized_datasets[split_name])
    y_true = predictions.label_ids
    y_pred = np.argmax(predictions.predictions, axis=-1)

    metrics = {
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }

    report_df = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            target_names=LABELS,
            output_dict=True,
            zero_division=0,
        )
    ).T
    cm_df = pd.DataFrame(confusion_matrix(y_true, y_pred), index=LABELS, columns=LABELS)

    report_df.to_csv(REPORT_DIR / f"{split_name}_classification_report.csv", encoding="utf-8-sig")
    cm_df.to_csv(REPORT_DIR / f"{split_name}_confusion_matrix.csv", encoding="utf-8-sig")

    return metrics


validation_metrics = evaluate_split("validation")
test_metrics = evaluate_split("test")

all_metrics = {
    "model_name": MODEL_NAME,
    "target_col": TARGET_COL,
    "labels": LABELS,
    "max_length": MAX_LENGTH,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "baseline_test_macro_f1": 0.811936,
    "validation": validation_metrics,
    "test": test_metrics,
    "model_dir": str(MODEL_DIR),
}

with open(REPORT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, ensure_ascii=False, indent=2)

with open(MODEL_ROOT / "xlmr_sentiment_latest.txt", "w", encoding="utf-8") as f:
    f.write(str(MODEL_DIR))

print(json.dumps(all_metrics, ensure_ascii=False, indent=2))
print("Reports:", REPORT_DIR)

{
  "model_name": "FacebookAI/xlm-roberta-base",
  "target_col": "sentiment_std_3class",
  "labels": [
    "negative",
    "neutral",
    "positive"
  ],
  "max_length": 192,
  "train_batch_size": 16,
  "eval_batch_size": 32,
  "num_epochs": 3,
  "learning_rate": 2e-05,
  "baseline_test_macro_f1": 0.811936,
  "validation": {
    "split": "validation",
    "accuracy": 0.8666121112929623,
    "macro_f1": 0.8639911477924865,
    "weighted_f1": 0.8665495381616733
  },
  "test": {
    "split": "test",
    "accuracy": 0.8545863585233664,
    "macro_f1": 0.8520639182987177,
    "weighted_f1": 0.8546099997699863
  },
  "model_dir": "/content/drive/MyDrive/Student Voice Intelligence/outputs/models/transformer/xlmr_sentiment_20260619_193243/model"
}
Reports: /content/drive/MyDrive/Student Voice Intelligence/outputs/models/transformer/xlmr_sentiment_20260619_193243/reports
